In [ ]:
# Fit FoSTA and plot the aligned tree domains

import sys, pathlib

sys.path.insert(0, str(next(p for p in [pathlib.Path.cwd()] + list(pathlib.Path.cwd().parents) if (p/"src").is_dir())))

import numpy as np
import matplotlib.pyplot as plt
from src.fosta import FoSTA
from src.pamona import Pamona
from src.kemarbf import KEMArbf
from src.mali import MALI
from utils.tree_utils import gen_tree


# ---------------------------------------------------------------------
# Generate two tree-like domains with partial label overlap
# ---------------------------------------------------------------------
X_a, y_a, _ = gen_tree(
    n_branch=8,
    n_child=2,
    n_dim_per_branch=4,
    branch_length=100,
    seed=1,
    sigma=4,
    merged_branch=False,
)

X_b, y_b, _ = gen_tree(
    n_branch=10,  # one extra class/topological branch in B
    n_child=2,
    n_dim_per_branch=4,
    branch_length=100,
    seed=2,
    sigma=4,
    merged_branch=False,
)

# ---------------------------------------------------------------------
# Optional partial labelling in target domain B
# ---------------------------------------------------------------------
MASK_TARGET_LABELS = False
TARGET_MASK_FRACTION = 0.5
MASK_SEED = 0
UNLABELED_VALUE = -1

y_b_obs = y_b.copy()

if MASK_TARGET_LABELS:
    rng = np.random.default_rng(MASK_SEED)

    shared_labels = np.intersect1d(
        np.unique(y_a),
        np.unique(y_b),
    )

    idx_shared_b = np.flatnonzero(np.isin(y_b_obs, shared_labels))

    idx_mask = rng.choice(
        idx_shared_b,
        size=int(TARGET_MASK_FRACTION * len(idx_shared_b)),
        replace=False,
    )

    y_b_obs[idx_mask] = UNLABELED_VALUE

    print(
        f"Masked {len(idx_mask)}/{len(idx_shared_b)} target labels "
        f"from shared classes only ({TARGET_MASK_FRACTION:.0%})."
    )

# ---------------------------------------------------------------------
# Fit FoSTA
# ---------------------------------------------------------------------
# model = FoSTA(
#     n_estimators=200,
#     random_state=0,
#     verbose=1,
#     ot_solver="hiref",
#     embedder="PHATE",
#     n_components=2,
#     unlabeled_coupling="include",  # use "exclude" or "predict_shared" to test behavior
# )

model = KEMArbf(random_state=0, verbose=1, n_components=2)

# model = MALI(random_state=0, verbose=1, n_components=2)
# model = Pamona(random_state=0, verbose=1, n_components=2)


Z = model.fit_transform(X_a, X_b, y_a, y_b_obs)

# ---------------------------------------------------------------------
# Plot by domain and by label
# ---------------------------------------------------------------------
n_a = X_a.shape[0]
domain = np.array(["A"] * n_a + ["B"] * X_b.shape[0])
labels = np.concatenate([y_a, y_b_obs])

fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)

# Domain plot
for dom in ["A", "B"]:
    idx = domain == dom
    axes[0].scatter(
        Z[idx, 0],
        Z[idx, 1],
        s=8,
        alpha=0.75,
        label=f"Domain {dom}",
    )

axes[0].set_title("FoSTA embedding colored by domain")
axes[0].set_xticks([])
axes[0].set_yticks([])
axes[0].legend(frameon=True, markerscale=2)

# Label plot
for lab in np.unique(labels):
    idx = labels == lab
    axes[1].scatter(
        Z[idx, 0],
        Z[idx, 1],
        s=8,
        alpha=0.75,
        label=f"Class {lab}",
    )

axes[1].set_title("FoSTA embedding colored by label")
axes[1].set_xticks([])
axes[1].set_yticks([])
axes[1].legend(frameon=True, markerscale=2, bbox_to_anchor=(1.02, 1), loc="upper left")

plt.show()